# Factual Consistency Method Comparison

This notebook compares 3 reranking methods:
1. **Baseline** — top-1 beam search (no reranking)
2. **NLI Reranking** — reranking based on entailment score
3. **Semantic Similarity Reranking** — reranking based on cosine similarity (sentence-transformers)

## 1. Setup

In [1]:
import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

!curl -s --max-time 5 https://huggingface.co > /dev/null && echo "Internet: ON" || echo "Internet: OFF"

!pip install -q transformers evaluate rouge-score sentence-transformers

GPU available: True
GPU name: Tesla T4
Internet: ON
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 95.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have n

## 2. Load Models & Data

In [2]:
import os
import json
import math
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForSequenceClassification
from sentence_transformers import SentenceTransformer, util

MODEL_PATH = "./checkpoints/bart-baseline"
TEST_FILE = "./data/test.jsonl"
TEXT_COLUMN = "article"
SUMMARY_COLUMN = "summary"
NLI_MODEL_NAME = "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load BART
bart_tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
bart_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_PATH).to(device).eval()
print("BART loaded")

# Load NLI
nli_tokenizer = AutoTokenizer.from_pretrained(NLI_MODEL_NAME)
nli_model = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL_NAME).to(device).eval()
id2label = {int(k): v.lower() for k, v in nli_model.config.id2label.items()}
ent_idx = next(i for i, l in id2label.items() if "entail" in l)
con_idx = next(i for i, l in id2label.items() if "contrad" in l)
neu_idx = next(i for i, l in id2label.items() if "neutral" in l)
print("NLI loaded")

# Load Sentence Transformer
sim_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2", device=device)
print("Sentence Transformer loaded")

# Load test data
with open(TEST_FILE, "r", encoding="utf-8") as f:
    test_rows = [json.loads(l) for l in f if l.strip()]
print(f"Test samples: {len(test_rows)}")

Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

BART loaded


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spm.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Sentence Transformer loaded
Test samples: 10972


## 3. Generate Candidates + Scoring

Generate 4 candidates per sample, then score with NLI and BERTScore.

In [3]:
NUM_CANDIDATES = 4
all_candidates = []

for i, row in enumerate(test_rows):
    document = row[TEXT_COLUMN]

    # Generate candidates
    with torch.inference_mode():
        inputs = bart_tokenizer(document, truncation=True, max_length=256, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}
        generated = bart_model.generate(
            **inputs,
            max_length=128, min_length=32,
            num_beams=NUM_CANDIDATES, num_return_sequences=NUM_CANDIDATES,
            length_penalty=1.0, early_stopping=True,
            output_scores=True, return_dict_in_generate=True,
        )
    texts = bart_tokenizer.batch_decode(generated.sequences, skip_special_tokens=True, clean_up_tokenization_spaces=True)
    scores = generated.sequences_scores.tolist()

    # NLI score each candidate
    candidates = []
    for text, gen_score in zip(texts, scores):
        summary = text.strip()
        with torch.inference_mode():
            enc = nli_tokenizer(document, summary, truncation=True, max_length=512, return_tensors="pt")
            enc = {k: v.to(device) for k, v in enc.items()}
            probs = torch.softmax(nli_model(**enc).logits[0], dim=-1)
        candidates.append({
            "summary": summary,
            "generation_score": float(gen_score),
            "entailment": float(probs[ent_idx]),
            "contradiction": float(probs[con_idx]),
        })

    # Semantic similarity for each candidate
    doc_emb = sim_model.encode(document, convert_to_tensor=True)
    for c in candidates:
        sum_emb = sim_model.encode(c["summary"], convert_to_tensor=True)
        c["sim_score"] = float(util.cos_sim(doc_emb, sum_emb)[0][0])

    all_candidates.append({
        "id": row.get("id"),
        "document": document,
        "reference_summary": row.get(SUMMARY_COLUMN),
        "candidates": candidates,
    })

    if (i + 1) % 200 == 0:
        print(f"Generated {i+1}/{len(test_rows)}", flush=True)

print(f"\nDone. {len(all_candidates)} samples processed.")

Generated 200/10972
Generated 400/10972
Generated 600/10972
Generated 800/10972
Generated 1000/10972
Generated 1200/10972
Generated 1400/10972
Generated 1600/10972
Generated 1800/10972
Generated 2000/10972
Generated 2200/10972
Generated 2400/10972
Generated 2600/10972
Generated 2800/10972
Generated 3000/10972
Generated 3200/10972
Generated 3400/10972
Generated 3600/10972
Generated 3800/10972
Generated 4000/10972
Generated 4200/10972
Generated 4400/10972
Generated 4600/10972
Generated 4800/10972
Generated 5000/10972
Generated 5200/10972
Generated 5400/10972
Generated 5600/10972
Generated 5800/10972
Generated 6000/10972
Generated 6200/10972
Generated 6400/10972
Generated 6600/10972
Generated 6800/10972
Generated 7000/10972
Generated 7200/10972
Generated 7400/10972
Generated 7600/10972
Generated 7800/10972
Generated 8000/10972
Generated 8200/10972
Generated 8400/10972
Generated 8600/10972
Generated 8800/10972
Generated 9000/10972
Generated 9200/10972
Generated 9400/10972
Generated 9600/10

## 4. Reranking & Evaluation

In [4]:
import statistics
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=False)

ALPHA = 0.7

methods = {
    "Baseline (top-1 beam)": lambda candidates: max(candidates, key=lambda c: c["generation_score"]),
    "NLI Reranking": lambda candidates: max(candidates, key=lambda c: ALPHA * c["entailment"] + (1 - ALPHA) * math.tanh(c["generation_score"] / 10.0)),
    "Semantic Similarity": lambda candidates: max(candidates, key=lambda c: c["sim_score"]),
}

results = []

for method_name, select_fn in methods.items():
    r1, r2, rl = [], [], []
    ent_scores, con_scores, sim_scores = [], [], []

    for item in all_candidates:
        best = select_fn(item["candidates"])

        s = scorer.score(item["reference_summary"], best["summary"])
        r1.append(s['rouge1'].fmeasure)
        r2.append(s['rouge2'].fmeasure)
        rl.append(s['rougeL'].fmeasure)
        ent_scores.append(best["entailment"])
        con_scores.append(best["contradiction"])
        sim_scores.append(best["sim_score"])

    n = len(all_candidates)
    result = {
        "method": method_name,
        "rouge1": sum(r1) / n,
        "rouge2": sum(r2) / n,
        "rougeL": sum(rl) / n,
        "entailment": statistics.mean(ent_scores),
        "contradiction": statistics.mean(con_scores),
        "sim_score": statistics.mean(sim_scores),
    }
    results.append(result)

print("=== Method Comparison ===")
print(f"{'Method':<25} | {'R-1':>6} | {'R-2':>6} | {'R-L':>6} | {'Entail':>7} | {'Contra':>7} | {'SimScore':>8}")
print("-" * 88)
for r in results:
    print(f"{r['method']:<25} | {r['rouge1']:.4f} | {r['rouge2']:.4f} | {r['rougeL']:.4f} | {r['entailment']:.4f}  | {r['contradiction']:.4f}  | {r['sim_score']:.4f}")

=== Method Comparison ===
Method                    |    R-1 |    R-2 |    R-L |  Entail |  Contra | SimScore
----------------------------------------------------------------------------------------
Baseline (top-1 beam)     | 0.3857 | 0.2122 | 0.3176 | 0.3100  | 0.0138  | 0.7707
NLI Reranking             | 0.3833 | 0.2084 | 0.3156 | 0.4662  | 0.0113  | 0.7658
Semantic Similarity       | 0.3916 | 0.2135 | 0.3189 | 0.3320  | 0.0141  | 0.8102


## 5. Save Results

In [ ]:
output_path = Path("./results/comparison_methods_results.json")
output_path.parent.mkdir(parents=True, exist_ok=True)

with output_path.open("w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"Saved to {output_path}")